In [3]:
# ============================================================
# PUNJABI NEWS CLASSIFICATION — SEED 42 CPU INFERENCE
# Works in Kaggle (CPU), Jupyter or Colab (CPU).
# IMPORTANT: change ARTIFACT_ROOT to your saved model's folder.
# This cell loads an existing fitted model; it does NOT train.
# ============================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"       # Force CPU before importing TensorFlow
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

from pathlib import Path
import json, pickle, re
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import tokenizer_from_json

# ONE PATH TO EDIT. Example: an attached Kaggle Dataset containing
# model.keras, tokenizer.json, tfidf_vectorizer.pkl, label_mapping.json
ARTIFACT_ROOT = Path("/kaggle/input/your-seed42-model")

# You can also point here instead:
# ARTIFACT_ROOT = Path("/kaggle/working/HySCF_SixFusion_5Seed_Components")
# ARTIFACT_ROOT = Path("/kaggle/working/Punjabi-News-Classification")

TITLE = "ਪੰਜਾਬ ਵਿੱਚ ਕਿਸਾਨਾਂ ਲਈ ਨਵੀਂ ਖੇਤੀਬਾੜੀ ਯੋਜਨਾ ਦਾ ਐਲਾਨ"
CONTENT = "ਸਰਕਾਰ ਨੇ ਕਿਸਾਨਾਂ ਦੀ ਮਦਦ ਲਈ ਨਵੀਂ ਯੋਜਨਾ ਬਾਰੇ ਜਾਣਕਾਰੀ ਦਿੱਤੀ ਹੈ।"

# Locate files in EITHER the original Kaggle output structure
# OR the flat experiments/ folder in the simplified GitHub release.
original_experiment = ARTIFACT_ROOT / "experiments" / "F06_PARALLEL_FLAT_CNN__seed42"
flat_experiment = ARTIFACT_ROOT / "experiments"
if (ARTIFACT_ROOT / "model.keras").is_file():
    model_dir = ARTIFACT_ROOT
elif (original_experiment / "model.keras").is_file():
    model_dir = original_experiment
elif (flat_experiment / "model.keras").is_file():
    model_dir = flat_experiment
else:
    raise FileNotFoundError(
        f"model.keras not found under {ARTIFACT_ROOT}. "
        "Attach the saved seed-42 artifacts as a Kaggle Dataset, "
        "or specify their actual location in ARTIFACT_ROOT."
    )

preprocessors_dir = (ARTIFACT_ROOT / "preprocessors") if (ARTIFACT_ROOT / "preprocessors").is_dir() else model_dir

def first_existing(*candidates):
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Missing required preprocessor: " + " OR ".join(str(p) for p in candidates))

tok_path = first_existing(preprocessors_dir / "tokenizer.json", model_dir / "tokenizer.json")
vec_path = first_existing(
    preprocessors_dir / "tfidf_5000" / "vectorizer.pkl",
    preprocessors_dir / "tfidf_vectorizer.pkl",
    model_dir / "tfidf_vectorizer.pkl"
)
classes_path = first_existing(
    preprocessors_dir / "label_mapping.json",
    model_dir / "label_mapping.json"
)

# Only load pickle files from YOUR original training or a trusted source.
print("TensorFlow:", tf.__version__, "| CPU GPUs visible:", tf.config.list_physical_devices("GPU"))
print("Model:", model_dir / "model.keras")
print("Tokenizer:", tok_path)
print("TF-IDF:", vec_path)

with tok_path.open(encoding="utf-8") as f:
    tokenizer = tokenizer_from_json(f.read())
with vec_path.open("rb") as f:
    vectorizer = pickle.load(f)
with classes_path.open(encoding="utf-8") as f:
    classes = json.load(f)["classes"]

model = tf.keras.models.load_model(model_dir / "model.keras", compile=False)
input_names = {tensor.name.split(":")[0] for tensor in model.inputs}
if input_names != {"han_input", "tfidf_input"}:
    raise ValueError(f"Unexpected model inputs: {input_names}. Need selected three-branch F06 model.")
if model.count_params() != 3_417_100:
    raise ValueError(f"Wrong model: {model.count_params()} parameters instead of 3,417,100.")
if len(classes) != 10:
    raise ValueError(f"Expected ten categories, found {len(classes)}")

card_path = model_dir / "MODEL_CARD.json"
if card_path.is_file():
    card = json.loads(card_path.read_text(encoding="utf-8"))
    if int(card.get("seed", -1)) != 42 or int(card.get("tfidf_features", -1)) != 5000:
        raise ValueError("Wrong model card: select seed 42 and TF-IDF 5000.")

# EXACT preprocessing from the original experiment: 
# 15 punctuation-delimited segments x 30 tokenizer IDs.
def make_han(text):
    output = np.zeros((1, 15, 30), dtype=np.int32)
    segments = re.split(r"[।!?]", text)[:15]
    sequences = tokenizer.texts_to_sequences(segments)
    for j, token_ids in enumerate(sequences):
        token_ids = token_ids[:30]
        output[0, j, :len(token_ids)] = token_ids
    return output

def predict_news(title, content):
    text = (str(title) + " " + str(content)).strip()
    if not text:
        raise ValueError("Enter a Punjabi headline or article text.")
    x_context = make_han(text)
    x_lexical = vectorizer.transform([text]).toarray().astype(np.float32)
    if x_lexical.shape != (1, 5000):
        raise ValueError(f"Wrong TF-IDF vectorizer: expected (1, 5000), got {x_lexical.shape}")
    prediction = np.asarray(model({
        "han_input": x_context,
        "tfidf_input": x_lexical
    }, training=False), dtype=np.float32)
    if prediction.shape != (1, 10) or not np.isfinite(prediction).all():
        raise ValueError(f"Unexpected model output: {prediction.shape}")
    if not np.isclose(prediction.sum(), 1.0, atol=1e-4):
        raise ValueError("Model did not return valid softmax probabilities")
    scores = prediction[0]
    top = np.argsort(scores)[::-1]
    print("\nPredicted category:", classes[int(top[0])])
    print("Model confidence: {:.2f}%".format(100 * float(scores[top[0]])))
    print("\nTop five categories:")
    for index in top[:5]:
        print("  {:15s} {:6.2f}%".format(str(classes[int(index)]), 100 * float(scores[index])))
    return {"label": classes[int(top[0])],
            "confidence": float(scores[top[0]]),
            "probabilities": {str(classes[i]): float(scores[i]) for i in range(10)}}

result = predict_news(TITLE, CONTENT)


Keras: 2.15.0
TensorFlow: 2.15.0


RuntimeError: This model requires Keras 3.13.2. Select your compatible Jupyter kernel and rerun.

In [2]:
pip install "tensorflow==2.20.0" "keras==3.11.3"

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\gurpr\\AppData\\Local\\Programs\\Python\\Python310\\Lib\\site-packages\\google\\~upb\\_message.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



  Obtaining dependency information for tensorflow==2.20.0 from https://files.pythonhosted.org/packages/74/b5/d40e1e389e07de9d113cf8e5d294c04d06124441d57606febfd0fb2cf5a6/tensorflow-2.20.0-cp310-cp310-win_amd64.whl.metadata
  Obtaining dependency information for keras==3.11.3 from https://files.pythonhosted.org/packages/94/5b/4c778cc921ce4b864b238f63f8e3ff6e954ab19b80c9fa680593ad8093d4/keras-3.11.3-py3-none-any.whl.metadata
  Obtaining dependency information for protobuf>=5.28.0 from https://files.pythonhosted.org/packages/8a/55/b77bda4e5e5f5971fb51b07663694690e9afdb9402136c16a522bd621cad/protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata
  Obtaining dependency information for tensorboard~=2.20.0 from https://files.pythonhosted.org/packages/9c/d9/a5db55f88f258ac669a92858b70a714bbbd5acd993820b41ec4a96a4d77f/tensorboard-2.20.0-py3-none-any.whl.metadata
  Obtaining dependency information for h5py>=3.11.0 from https://files.pythonhosted.org/packages/66/35/d88fd6718832133c885004c61ceeeb24dbd